In [21]:
import cv2
import mediapipe as mp
import numpy as np
import os

In [30]:
mp_pose = mp.solutions.pose
pose = tmp_pose.Pose()
mp_drawing = mp.solutions.drawing_utils

I0000 00:00:1735828425.140533 14814380 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 83), renderer: Apple M2
W0000 00:00:1735828425.215924 14825933 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1735828425.227836 14825935 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [31]:
def calculate_angle(a, b, c):
    a = np.array(a)  # First point
    b = np.array(b)  # Midpoint
    c = np.array(c)  # End point

    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)

    if angle > 180.0:
        angle = 360.0 - angle

    return angle

In [38]:
def analyze_squaring_up(image, landmarks, side="right"):
    feedback = []

    # Dynamically choose landmarks
    shoulder = landmarks[getattr(mp_pose.PoseLandmark, f"{side.upper()}_SHOULDER").value]
    elbow = landmarks[getattr(mp_pose.PoseLandmark, f"{side.upper()}_ELBOW").value]
    wrist = landmarks[getattr(mp_pose.PoseLandmark, f"{side.upper()}_WRIST").value]
    hip = landmarks[getattr(mp_pose.PoseLandmark, f"{side.upper()}_HIP").value]
    knee = landmarks[getattr(mp_pose.PoseLandmark, f"{side.upper()}_KNEE").value]
    ankle = landmarks[getattr(mp_pose.PoseLandmark, f"{side.upper()}_ANKLE").value]

    # Elbow angle
    elbow_angle = calculate_angle([shoulder.x, shoulder.y], [elbow.x, elbow.y], [wrist.x, wrist.y])
    cv2.putText(image, f"Elbow: {int(elbow_angle)}", (int(elbow.x * image.shape[1]), int(elbow.y * image.shape[0])),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
    if elbow_angle < 85 or elbow_angle > 95:
        feedback.append(f"The player's elbow should be closer to a 90-degree angle.")

    # Knee angle
    knee_angle = calculate_angle([hip.x, hip.y], [knee.x, knee.y], [ankle.x, ankle.y])
    cv2.putText(image, f"Knee: {int(knee_angle)}", (int(knee.x * image.shape[1]), int(knee.y * image.shape[0])),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
    if knee_angle > 160:
        feedback.append(f"The player's knee should be bent below 160 degrees.")

    # Back alignment
    back_angle = calculate_angle([shoulder.x, shoulder.y], [hip.x, hip.y], [knee.x, knee.y])
    cv2.putText(image, f"Back: {int(back_angle)}", (int(hip.x * image.shape[1]), int(hip.y * image.shape[0])),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
    if back_angle < 170:
        feedback.append(f"The player's back should be straighter.")

    return feedback, image

In [39]:
def analyze_release(image, landmarks, side="right"):
    feedback = []

    # Dynamically choose landmarks
    shoulder = landmarks[getattr(mp_pose.PoseLandmark, f"{side.upper()}_SHOULDER").value]
    elbow = landmarks[getattr(mp_pose.PoseLandmark, f"{side.upper()}_ELBOW").value]
    wrist = landmarks[getattr(mp_pose.PoseLandmark, f"{side.upper()}_WRIST").value]
    hip = landmarks[getattr(mp_pose.PoseLandmark, f"{side.upper()}_HIP").value]
    knee = landmarks[getattr(mp_pose.PoseLandmark, f"{side.upper()}_KNEE").value]
    ankle = landmarks[getattr(mp_pose.PoseLandmark, f"{side.upper()}_ANKLE").value]

    # Arm straightness
    arm_angle = calculate_angle([shoulder.x, shoulder.y], [elbow.x, elbow.y], [wrist.x, wrist.y])
    cv2.putText(image, f"Arm: {int(arm_angle)}", (int(elbow.x * image.shape[1]), int(elbow.y * image.shape[0])),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
    if arm_angle > 10:
        feedback.append(f"The player's arm should be straighter during the release.")

    # Leg straightness
    leg_angle = calculate_angle([hip.x, hip.y], [knee.x, knee.y], [ankle.x, ankle.y])
    cv2.putText(image, f"Leg: {int(leg_angle)}", (int(knee.x * image.shape[1]), int(knee.y * image.shape[0])),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
    if leg_angle < 170:
        feedback.append(f"The player's leg should be straighter during the release.")

    # Back alignment
    back_angle = calculate_angle([shoulder.x, shoulder.y], [hip.x, hip.y], [knee.x, knee.y])
    cv2.putText(image, f"Back: {int(back_angle)}", (int(hip.x * image.shape[1]), int(hip.y * image.shape[0])),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
    if back_angle < 170:
        feedback.append(f"The player's back should be straighter.")

    return feedback, image

In [40]:
squared_up_path = "results/squared_up_frame.jpg"
release_path = "results/release_frame.jpg"

In [41]:
side = "right"  

In [42]:
if os.path.exists(squared_up_path):
    squared_up_image = cv2.imread(squared_up_path)
    result = pose.process(cv2.cvtColor(squared_up_image, cv2.COLOR_BGR2RGB))

    if result.pose_landmarks:
        feedback, analyzed_image = analyze_squaring_up(squared_up_image, result.pose_landmarks.landmark, side)
        cv2.imwrite("results/squared_up_feedback.jpg", analyzed_image)
        print("Squaring Up Feedback:")
        for item in feedback:
            print(item)

Squaring Up Feedback:
The player's elbow should be closer to a 90-degree angle.
The player's back should be straighter.


In [43]:
if os.path.exists(release_path):
    release_image = cv2.imread(release_path)
    result = pose.process(cv2.cvtColor(release_image, cv2.COLOR_BGR2RGB))

    if result.pose_landmarks:
        feedback, analyzed_image = analyze_release(release_image, result.pose_landmarks.landmark, side)
        cv2.imwrite("results/release_feedback.jpg", analyzed_image)
        print("\nRelease Feedback:")
        for item in feedback:
            print(item)